### Calculate fractional coverage (weights) for pixel intersecting a polygon layer
Inputs:
- Polygon layer
- Grid (ie, netcdf file)
Output:
- Writes to weights table in the warehouse
- Table name: "grid_pixel_coverage_weights"
- Schema:
  - `fraction_covered`: double
  - `position_index`: int
  - `location_id`: string (this represents the polygon layer)
  - `configuration_name`: string (represents the the grid model configuration)
  - `variable_name`: string (represents the grid variable)
  - `row`: int
  - `col`: int
  - `domain_name`: string (represents the domain of the model, ex: 'nwm30_conus_forcing'. Defines uniqueness together with location_id)

In [ ]:
import geopandas as gpd
from pyspark.sql import functions as F
import botocore.session
import os
from pathlib import Path

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
# # These were read directly from sample netcdf files
# NWM_CONUS_CRS = "+proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1=30.0 +lat_2=60.0 +lat_0=40.0 +lon_0=-97.0 +x_0=0 +y_0=0 +k_0=1.0 +nadgrids=@null +wktext  +no_defs"
# NWM_AK_CRS = "+proj=stere +lat_0=90 +lat_ts=60 +lon_0=-135 +x_0=0 +y_0=0 +R=6370000 +units=m +no_defs"
# NWM_HI_CRS = "+proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1=10.0 +lat_2=30.0 +lat_0=20.6 +lon_0=-157.42 +x_0=0 +y_0=0 +k_0=1.0 +nadgrids=@null +wktext  +no_defs "
# NWM_PR_CRS = "+proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1=18.1 +lat_2=18.1 +lat_0=18.1 +lon_0=-65.91 +x_0=0 +y_0=0 +k_0=1.0 +nadgrids=@null +wktext  +no_defs "

In [ ]:
# CONFIGURATION_NAME = "nwm30_retrospective"
# # CONFIGURATION_NAME = "nwm30_forcing_alaska"
# VARIABLE_NAME = "rainfall_hourly_rate"
# TABLE_NAME = "grid_pixel_coverage_weights"

# # TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/197902010000.LDASIN_DOMAIN1"  # NWM v3.0 retrospective
# TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t00z.short_range.forcing.f003.conus.nc"  # NWM v3.0 forcing short range 
# # TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t01z.analysis_assim.forcing.tm00.alaska.nc"
# # TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t00z.short_range.forcing.f003.hawaii.nc"
# # TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t06z.short_range.forcing.f004.puertorico.nc"

In [ ]:
%%time
spark = create_spark_session(
    aws_profile="admin-user",
    start_spark_cluster=True,
    executor_instances=8,
    executor_memory="32g",
    executor_cores=4,
    update_configs={
        # "spark.sql.shuffle.partitions": 512,
        "spark.sql.autoBroadcastJoinThreshold": "300mb"
    }
)

ev = teehr.RemoteReadWriteEvaluation(spark=spark)
# ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [ ]:
existing_grid_table = ev.table(table_name="grid_pixel_coverage_weights").to_sdf()
# existing_grid_table.select("configuration_name").distinct().show(truncate=False)

In [ ]:
existing_grid_table.show(n=5, truncate=False)

In [ ]:
updated_grid_table = existing_grid_table.withColumn("domain_name", F.lit("nwm30_conus_forcing"))

In [ ]:
updated_grid_table.show(n=5, truncate=False)

In [ ]:
%%time
ev._write.to_warehouse(
    source_data=updated_grid_table,
    table_name="grid_pixel_coverage_weights",
    partition_by=["domain_name"],
    write_mode="create_or_replace"
)

### Read in the polygons for weights calculation.

We need to reproject the polygon layer to match the NWM v3.0 retro CRS. The proj4 string below was extracted from the Zarr attributes at "s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/forcing/precip.zarr"

In [ ]:
BASIN_POLYGON_LAYER = Path("/data/common/geometry/usgs_basins/usgsbasin_geometry_04202026.parquet")

# nmw30_proj4 = "+proj=lcc +units=m +a=6370000.0 +b=6370000.0 +lat_1=30.0 +lat_2=60.0 +lat_0=40.0 +lon_0=-97.0 +x_0=0 +y_0=0 +k_0=1.0 +nadgrids=@null +wktext  +no_defs"
# reproj_path = BASIN_POLYGON_LAYER.parent / Path(BASIN_POLYGON_LAYER.stem + "_nwm30crs.parquet")
# reproj_path

One time: Reproject the basins layer to the NWM CRS and repair any invalid geometries

In [ ]:
# gdf = gpd.read_parquet(BASIN_POLYGON_LAYER)
# gdf_nwm = gdf.to_crs(crs=nmw30_proj4)
# gdf_nwm["geometry"] = gdf_nwm.geometry.make_valid()  # NOTE: Had some invalid geometries. Always include this.
# gdf_nwm.to_parquet(reproj_path)  # Save the reprojected and validated version to disk.

In [ ]:
# gdf = gpd.read_parquet(reproj_path)

In [ ]:
# mask = gdf["id"].isin(pr_ids_df["id"].str.replace("usgs-", "usgsbasin-").tolist())
# gdf_ak = gdf[mask]
# gdf_ak = gdf_ak.to_crs(crs=NWM_AK_CRS)
# gdf_ak.to_parquet(BASIN_POLYGON_LAYER.parent / Path(BASIN_POLYGON_LAYER.stem + "_nwm30_alaska_crs.parquet"))

Load the polygons into a spark dataframe

In [ ]:
# # Alaska
# basin_filepath = BASIN_POLYGON_LAYER.parent / Path(BASIN_POLYGON_LAYER.stem + "_nwm30_alaska_crs_in_domain_only.parquet")
# TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t01z.analysis_assim.forcing.tm00.alaska.nc"
# domain_name = "nwm30_alaska_forcing"
# configuration_name = "nwm30_forcing_analysis_assim_alaska"

# # Hawaii
# basin_filepath = BASIN_POLYGON_LAYER.parent / Path(BASIN_POLYGON_LAYER.stem + "_nwm30_hawaii_crs.parquet")
# TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t00z.short_range.forcing.f003.hawaii.nc"
# domain_name = "nwm30_hawaii_forcing"
# configuration_name = "nwm30_forcing_short_range_hawaii"


# Puerto Rico
basin_filepath = BASIN_POLYGON_LAYER.parent / Path(BASIN_POLYGON_LAYER.stem + "_nwm30_puertorico_crs.parquet")
TEMPLATE_NETCDF_FILEPATH = "/data/playground/slamont/teehr/warehouse/sedona/nwm_operational/nwm.t06z.short_range.forcing.f004.puertorico.nc"
domain_name = "nwm30_puertorico_forcing"
configuration_name = "nwm30_forcing_short_range_puertorico"


variable_name = "rainfall_hourly_rate"
basin_filepath

In [ ]:
poly_sdf = (
    spark
    .read
    .format("parquet")
    .load(basin_filepath.as_posix())
    .selectExpr("ST_GeomFromWKB(geometry) as geometry", "id")
)

### Read in a single grid file as a template

In [ ]:
raster_sdf = spark.read.format("binaryFile").load(TEMPLATE_NETCDF_FILEPATH).selectExpr("RS_FromNetCDF(content, 'RAINRATE', 'x', 'y') as raster", "path as filepath") 

In [ ]:
raster_geom_sdf = raster_sdf.limit(1).selectExpr(
  "explode(RS_PixelAsPolygons(raster, 1)) as exploded"
).selectExpr(
  "exploded.geom as geom",
  "exploded.value as value",
  "exploded.x as col",
  "exploded.y as row"
)
raster_geom_sdf.show(3)

Calculate a 1-D "position index" that maps the row/col position using the grid dimensions (width)

In [ ]:
%%time
raster_width = raster_sdf.selectExpr("RS_Width(raster) as width", "RS_Height(raster) as height").collect()[0]["width"]

raster_geom_sdf = raster_geom_sdf.withColumn("position_index", ((F.col("row") - 1) * f"{raster_width}" + (F.col("col") - 1)))
raster_geom_sdf.show(3)

### Now we can calculate the weights and write to the warehouse

In [ ]:
# This does not add partitions for unique values, but rather some default number unless specified (17?)
raster_geom_sdf = raster_geom_sdf.repartition("geom")
poly_sdf = poly_sdf.repartition("id")

In [ ]:
poly_sdf.createOrReplaceTempView("polygon_view")
raster_geom_sdf.createOrReplaceTempView("raster_polygons_view")

In [ ]:
weights_results_sdf = spark.sql("""
    SELECT /*+ BROADCAST(pv) */
        ST_Area(ST_Intersection(rpv.geom, pv.geometry)) / ST_Area(rpv.geom) as fraction_covered,
        rpv.position_index, rpv.row, rpv.col, pv.id as location_id    
    FROM raster_polygons_view AS rpv
    INNER JOIN polygon_view AS pv
    ON ST_Intersects(rpv.geom, pv.geometry)
""")

In [ ]:
%%time
coverage_weights_sdf = weights_results_sdf.withColumnsRenamed(
    {
        "id": "location_id",
        "pos": "position_index"
    }
).withColumns(
    {
        "configuration_name": F.lit(configuration_name),
        "variable_name": F.lit(variable_name),
        "domain_name": F.lit(domain_name)
    }
)
coverage_weights_sdf.show(n=5, truncate=False)

#### Write to the warehouse

In [ ]:
%%time
ev._write.to_warehouse(
    source_data=coverage_weights_sdf,
    table_name="grid_pixel_coverage_weights",
    write_mode="append",
    uniqueness_fields=["location_id", "domain_name"]
)

In [ ]:
grid_table = ev.table(table_name="grid_pixel_coverage_weights").to_sdf()
grid_table.show(n=5, truncate=False)

In [ ]:
grid_table.select("domain_name").distinct().show(truncate=False)

In [ ]:
grid_table.filter("domain_name = 'nwm30_puertorico_forcing'").select("location_id").distinct().count()